# 00 Parking Sensor Data Ingestion

## Purpose
This notebook downloads the parking sensor datasets (2011–2020) required for the project and stores them in the project's `data/raw` folder. It also checks whether the datasets already exist to avoid unnecessary downloads. The downloaded datasets will be used in the data preparation and analysis notebooks.

The downloaded files will be used in:

- `01_sensor_data_preparation.ipynb`
- `02_sensor_cleaning_eda_duckdb.ipynb`

Large data files are stored locally and are not uploaded to GitHub.

### 1.  Import libraries
Import the Python libraries required to download datasets, manage project folders, and extract compressed files.

In [1]:
# Import libraries for downloading and managing files
from pathlib import Path
import urllib.request
import zipfile

### 2. Create the project raw data folder
Create the project's raw data folder if it does not already exist. This folder will store all downloaded parking sensor datasets.

In [2]:
# Project data folder
raw_dir = Path("../../../data/raw")

# Create the folder if it does not exist
raw_dir.mkdir(parents=True, exist_ok=True)

print(raw_dir.resolve())

/Users/thanya/Desktop/Victoria-Urban-Planning/data/raw


### 3. Define the parking sensor datasets
Store the download URL and expected file name for each parking sensor dataset. This allows the download process to be managed automatically for all years.

In [22]:
# Download information for parking sensor datasets from 2011 to 2020
parking_sensor_datasets = {
    2011: {
        "url": "https://opendatasoft-s3.s3.amazonaws.com/downloads/archive/vkxi-k7ps.zip",
        "csv_name": "On-street_Car_Parking_Sensor_Data_-_2011.csv",
    },
    2012: {
        "url": "https://opendatasoft-s3.s3.amazonaws.com/downloads/archive/vbe9-m4tk.zip",
        "csv_name": "On-street_Car_Parking_Sensor_Data_-_2012.csv",
    },
    2013: {
        "url": "https://opendatasoft-s3.s3.amazonaws.com/downloads/archive/7jq6-k9kf.zip",
        "csv_name": "On-street_Car_Parking_Sensor_Data_-_2013.csv",
    },
    2014: {
        "url": "https://opendatasoft-s3.s3.amazonaws.com/downloads/archive/t6hb-9uf2.zip",
        "csv_name": "On-street_Car_Parking_Sensor_Data_-_2014.csv",
    },
    2015: {
        "url": "https://opendatasoft-s3.s3.amazonaws.com/downloads/archive/apua-t2tb.zip",
        "csv_name": "On-street_Car_Parking_Sensor_Data_-_2015.csv",
    },
    2016: {
        "url": "https://opendatasoft-s3.s3.amazonaws.com/downloads/archive/dj7e-rdx9.zip",
        "csv_name": "On-street_Car_Parking_Sensor_Data_-_2016.csv",
    },
    2017: {
        "url": "https://opendatasoft-s3.s3.amazonaws.com/downloads/archive/u9sa-j86i.zip",
        "csv_name": "On-street_Car_Parking_Sensor_Data_-_2017.csv",
    },
    2018: {
        "url": "https://opendatasoft-s3.s3.amazonaws.com/downloads/archive/5532-ig9r.zip",
        "csv_name": "On-street_Car_Parking_Sensor_Data_-_2018.csv",
    },
    2019: {
        "url": "https://opendatasoft-s3.s3.amazonaws.com/downloads/archive/7pgd-bdf2.zip",
        "csv_name": "On-street_Car_Parking_Sensor_Data_-_2019.csv",
    },
    2020: {
    "url": "https://opendatasoft-s3.s3.amazonaws.com/downloads/archive/4n3a-s6rn.zip",
    "csv_name": "On-street_Car_Parking_Sensor_Data_-_2020__Jan_-_May_.csv",
    },
}

### 4. Download parking sensor datasets
Download any missing parking sensor datasets and skip files that already exist to avoid unnecessary downloads. 

In [23]:
# Download and extract all parking sensor datasets
for year, dataset in parking_sensor_datasets.items():

    csv_path = raw_dir / dataset["csv_name"]
    zip_path = raw_dir / f"parking_sensor_{year}.zip"

    # Skip the download if the CSV already exists
    if csv_path.exists():
        print(f"{year}: {dataset['csv_name']} already exists. Skipping.")
        continue

    print(f"{year}: Downloading...")

    try:
        urllib.request.urlretrieve(
            dataset["url"],
            zip_path
        )

        print(f"{year}: Download completed. Extracting...")

        with zipfile.ZipFile(zip_path, "r") as zip_ref:

            csv_files = [
                file_name
                for file_name in zip_ref.namelist()
                if file_name.lower().endswith(".csv")
            ]

            if not csv_files:
                raise FileNotFoundError(
                    f"No CSV file was found inside the {year} ZIP file."
                )

            extracted_path = Path(
                zip_ref.extract(
                    csv_files[0],
                    raw_dir
                )
            )

        # Rename the extracted CSV to the expected filename
        if extracted_path != csv_path:
            if csv_path.exists():
                csv_path.unlink()

            extracted_path.rename(csv_path)

        # Delete the ZIP after extraction to save storage
        if zip_path.exists():
            zip_path.unlink()

        print(f"{year}: Saved to {csv_path.name}")

    except Exception as error:
        print(f"{year}: Download failed - {error}")

        # Remove an incomplete ZIP file
        if zip_path.exists():
            zip_path.unlink()

2011: On-street_Car_Parking_Sensor_Data_-_2011.csv already exists. Skipping.
2012: On-street_Car_Parking_Sensor_Data_-_2012.csv already exists. Skipping.
2013: On-street_Car_Parking_Sensor_Data_-_2013.csv already exists. Skipping.
2014: On-street_Car_Parking_Sensor_Data_-_2014.csv already exists. Skipping.
2015: On-street_Car_Parking_Sensor_Data_-_2015.csv already exists. Skipping.
2016: On-street_Car_Parking_Sensor_Data_-_2016.csv already exists. Skipping.
2017: On-street_Car_Parking_Sensor_Data_-_2017.csv already exists. Skipping.
2018: On-street_Car_Parking_Sensor_Data_-_2018.csv already exists. Skipping.
2019: On-street_Car_Parking_Sensor_Data_-_2019.csv already exists. Skipping.
2020: On-street_Car_Parking_Sensor_Data_-_2020__Jan_-_May_.csv already exists. Skipping.


### 5. Validate downloaded datasets
Check that all required parking sensor datasets are available before continuing to the data preparation step.

In [24]:
# Check whether all expected parking sensor CSV files are available
download_status = []

for year, dataset in parking_sensor_datasets.items():
    csv_path = raw_dir / dataset["csv_name"]

    download_status.append(
        {
            "year": year,
            "file_name": dataset["csv_name"],
            "exists": csv_path.exists(),
            "file_path": str(csv_path),
        }
    )

for item in download_status:
    status = "Available" if item["exists"] else "Missing"
    print(f"{item['year']}: {status} - {item['file_name']}")

2011: Available - On-street_Car_Parking_Sensor_Data_-_2011.csv
2012: Available - On-street_Car_Parking_Sensor_Data_-_2012.csv
2013: Available - On-street_Car_Parking_Sensor_Data_-_2013.csv
2014: Available - On-street_Car_Parking_Sensor_Data_-_2014.csv
2015: Available - On-street_Car_Parking_Sensor_Data_-_2015.csv
2016: Available - On-street_Car_Parking_Sensor_Data_-_2016.csv
2017: Available - On-street_Car_Parking_Sensor_Data_-_2017.csv
2018: Available - On-street_Car_Parking_Sensor_Data_-_2018.csv
2019: Available - On-street_Car_Parking_Sensor_Data_-_2019.csv
2020: Available - On-street_Car_Parking_Sensor_Data_-_2020__Jan_-_May_.csv
